# 02 — Abstract traffic-simulation comparison

This notebook analyzes the provenance-validated abstract traffic benchmark. The network, routes, demand, controller plans, and emissions are simulated. These results are not observed Cairo congestion, a calibrated Tahrir model, or field-validated signal improvements.


## Load validated abstract-simulation artifacts

Run urbantwin traffic-run --resume only when the benchmark needs to be reproduced. This notebook never launches simulations; it reads the complete local bundle and rejects missing, incomplete, mismatched, or non-abstract artifacts.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urbantwin.visualization.report_writer import validate_traffic_artifacts

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Start Jupyter from the UrbanTwin-AI repository root.")

bundle = validate_traffic_artifacts(ROOT)
if bundle is None:
    raise FileNotFoundError(
        "Validated traffic artifacts are missing. Reproduce the benchmark before analysis."
    )
if bundle["provenance"].get("data_kind") != "simulation_abstract":
    raise ValueError("Notebook 02 accepts only the abstract simulation benchmark.")

metrics = bundle["metrics"].copy()
summary = bundle["summary"].copy()
paired_deltas = bundle["paired_deltas"].copy()
paired_summary = bundle["paired_summary"].copy()
provenance = bundle["provenance"]


## Experimental design and execution accounting

Every controller is evaluated on the same held-out seeds within each synthetic demand scenario. Optimizer-training seeds are separate from evaluation seeds; checkpoint cache hits do not alter the declared objective budgets.


In [ ]:
checkpointing = provenance.get("checkpointing", {})
design_summary = pd.Series(
    {
        "source label": provenance["data_kind"],
        "scenarios": ", ".join(sorted(metrics["scenario"].unique())),
        "methods": ", ".join(sorted(metrics["method"].unique())),
        "evaluation seeds per scenario/method": int(
            metrics.groupby(["scenario", "method"])["seed"].nunique().min()
        ),
        "optimizer/evaluation seed overlap": bool(metrics["train_seed_overlap"].any()),
        "logical executions completed": checkpointing.get(
            "completed_logical_executions", "not recorded"
        ),
        "physical executions in recorded invocation": checkpointing.get(
            "physical_executions_this_invocation", "not recorded"
        ),
        "checkpoint cache hits": checkpointing.get("cache_hits", "not recorded"),
        "simulator version": provenance.get("sumo_version", "not recorded"),
    },
    name="value",
)
design_summary.to_frame()


## Paired performance relative to fixed-time control

Deltas are method minus fixed-time for the same scenario and evaluation seed. Negative time-loss, waiting-time, or queue deltas are lower within this simulator; throughput and vehicle-count deltas require their own interpretation.


In [ ]:
paired_view = paired_summary[
    [
        "scenario",
        "method",
        "runs",
        "mean_delta_mean_time_loss_s_completed_method_minus_fixed",
        "std_delta_mean_time_loss_s_completed_method_minus_fixed",
        "mean_delta_mean_approach_queue_vehicles_method_minus_fixed",
        "mean_delta_completed_throughput_vehicles_per_sim_hour_method_minus_fixed",
        "mean_delta_unfinished_vehicles_method_minus_fixed",
        "mean_delta_pending_vehicles_method_minus_fixed",
    ]
].rename(
    columns={
        "mean_delta_mean_time_loss_s_completed_method_minus_fixed": "time-loss delta (s)",
        "std_delta_mean_time_loss_s_completed_method_minus_fixed": "time-loss delta SD (s)",
        "mean_delta_mean_approach_queue_vehicles_method_minus_fixed": "queue delta (vehicles)",
        "mean_delta_completed_throughput_vehicles_per_sim_hour_method_minus_fixed": "throughput delta (veh/h)",
        "mean_delta_unfinished_vehicles_method_minus_fixed": "unfinished delta",
        "mean_delta_pending_vehicles_method_minus_fixed": "pending delta",
    }
).sort_values(["scenario", "method"])
paired_view


In [ ]:
plot_rows = paired_view.reset_index(drop=True)
positions = np.arange(len(plot_rows))
plot_labels = plot_rows["scenario"] + " / " + plot_rows["method"]
figure, axis = plt.subplots(figsize=(11, 5))
axis.errorbar(
    positions,
    plot_rows["time-loss delta (s)"],
    yerr=plot_rows["time-loss delta SD (s)"],
    fmt="o",
    color="#2b6f8a",
    capsize=4,
)
axis.axhline(0, color="black", linewidth=1)
axis.set_xticks(positions, plot_labels, rotation=30, ha="right")
axis.set(
    ylabel="Completed-trip time-loss delta (s; method minus fixed)",
    title="Paired held-out-seed comparison in the abstract simulation",
)
axis.grid(axis="y", alpha=0.2)
figure.tight_layout()
plt.show()


Only three paired evaluation seeds are available per scenario and method. Mean ± standard deviation is descriptive uncertainty, not a significance test. A completed-trip average must be read alongside generated, inserted, completed, unfinished, and pending vehicles so horizon censoring is visible.


## Completion and censoring diagnostics


In [ ]:
completion_view = summary[
    [
        "scenario",
        "method",
        "runs",
        "mean_generated_vehicles",
        "mean_inserted_vehicles",
        "mean_completed_vehicles",
        "mean_unfinished_vehicles",
        "mean_pending_vehicles",
        "mean_completion_fraction_generated",
        "mean_completed_throughput_vph",
    ]
].sort_values(["scenario", "method"])
completion_view


In [ ]:
seed_sets = metrics.groupby(["scenario", "method"])["seed"].apply(
    lambda values: tuple(sorted(values.astype(int)))
)
checks = pd.Series(
    {
        "generated = inserted + pending": (
            metrics["generated_vehicles"]
            == metrics["inserted_vehicles"] + metrics["pending_vehicles"]
        ).all(),
        "inserted = completed + unfinished": (
            metrics["inserted_vehicles"]
            == metrics["completed_vehicles"] + metrics["unfinished_vehicles"]
        ).all(),
        "paired evaluation seeds": seed_sets.groupby(level="scenario").nunique().eq(1).all(),
        "training/evaluation seeds separated": (~metrics["train_seed_overlap"]).all(),
    },
    name="validated",
)
if not checks.all():
    raise ValueError("Traffic completion or seed-pairing checks failed.")
checks.to_frame()


## Simulator-modeled emissions

The emissions column integrates the simulator model for the inserted fleet under recorded vehicle-class assumptions. It is not observed environmental data, and lower modeled totals do not establish a real-world air-quality effect.


In [ ]:
emissions_view = summary[
    [
        "scenario",
        "method",
        "runs",
        "mean_inserted_vehicles",
        "mean_simulated_co2_g_inserted_fleet",
    ]
].sort_values(["scenario", "method"])
emissions_view


This benchmark compares algorithms under one abstract network, synthetic demand generator, fixed evaluation horizon, and three held-out seeds. It does not validate a controller on measured Cairo demand, quantify field performance, or support causal or statistically significant real-world claims.
